# Pseudo-Labeling — Unlabeled `train_soundscapes`

Uses `effnet_fold0_best.pth` to generate soft + hard labels for every
unlabeled soundscape. Audio preprocessing imports directly from
`birdclef_utils.audio` and `birdclef_utils.constants` — the model sees
exactly the same spectrogram distribution it was trained on.

Output:
- `pseudo_labels_raw.csv` — every 5-second window with raw per-class probabilities
- `pseudo_labels_filtered.csv` — hard labels after confidence threshold filtering
- `pseudo_labels_for_training.csv` — drop-in replacement for `labeled_soundscapes_split.csv`,
  ready to pass into `SoundscapeChunkDataset`

## 0 — Paths (edit these)

In [ ]:
import os

# ---------- adjust to your environment ----------
UTILS      = '/kaggle/input/datasets/tsapalyu/birdclef2026-utils'
COMP       = '/kaggle/input/competitions/birdclef-2026'
CHECKPOINT = '/kaggle/input/birdclef2026-utils/effnet_fold0_best.pth'  # your pretrained model
OUT_DIR    = '/kaggle/working'
# ------------------------------------------------

SS_DIR          = f'{COMP}/train_soundscapes'
LABELS_CSV      = f'{UTILS}/labeled_soundscapes_split.csv'  # already-labeled windows
LABEL2IDX_JSON  = f'{UTILS}/label2idx.json'

os.makedirs(OUT_DIR, exist_ok=True)
print("Paths OK")

## 1 — Identify unlabeled files

In [ ]:
import json
import pandas as pd

# All .ogg files in the soundscapes folder
all_ss_files = sorted(
    f for f in os.listdir(SS_DIR) if f.endswith('.ogg')
)

# Files that already have ground-truth labels
labels_df    = pd.read_csv(LABELS_CSV)
labeled_set  = set(labels_df['filename'].unique())

# Remaining files = pseudo-label targets
unlabeled_files = sorted(set(all_ss_files) - labeled_set)

print(f"Total soundscape files : {len(all_ss_files)}")
print(f"Already labeled        : {len(labeled_set)}")
print(f"To pseudo-label        : {len(unlabeled_files)}")
print(f"\nFirst 5 unlabeled:\n", unlabeled_files[:5])

## 2 — Load label mapping

In [ ]:
with open(LABEL2IDX_JSON) as f:
    label2idx = json.load(f)

idx2label = {v: k for k, v in label2idx.items()}
NUM_CLASSES = len(label2idx)
print(f"NUM_CLASSES = {NUM_CLASSES}")
print(f"First 5 labels: {list(label2idx.items())[:5]}")

## 3 — Imports from `birdclef_utils`

In [ ]:
import sys
import numpy as np
import librosa
import torch

sys.path.append(UTILS)

from birdclef_utils.constants import SR, DURATION, N_SAMPLES, N_MELS
from birdclef_utils.audio import (
    load_audio, normalize_waveform, fix_length, audio_to_melspec,
)


def load_chunk(filepath, start_sec):
    """Load a single 5-second chunk from a soundscape file.
    Returns a float32 tensor of shape (1, N_MELS, time_frames),
    or None if the file is unreadable.
    """
    try:
        y, _ = librosa.load(
            filepath, sr=SR, mono=True,
            offset=start_sec, duration=DURATION,
        )
        y = np.asarray(y, dtype=np.float32)
    except Exception as e:
        print(f"  load failed {filepath}@{start_sec}s: {e}")
        return None

    if y is None or y.size == 0:
        return None

    y = normalize_waveform(y)
    y = fix_length(y, target_len=N_SAMPLES, crop_mode='start')
    spec = audio_to_melspec(y)                      # (N_MELS, time_frames)
    return torch.from_numpy(spec).unsqueeze(0)      # (1, N_MELS, time_frames)


print(f"birdclef_utils imported. SR={SR}, DURATION={DURATION}, N_MELS={N_MELS}")
print(f"Expected spectrogram shape: (1, {N_MELS}, ~313) per chunk")

## 4 — Load model

In [ ]:
import timm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}")

# Build the architecture — must match training CONFIG exactly
model = timm.create_model(
    'tf_efficientnetv2_s.in21k_ft_in1k',
    pretrained=False,
    num_classes=NUM_CLASSES,
    in_chans=1,
)

# Load weights
ckpt = torch.load(CHECKPOINT, map_location=DEVICE)

# Support both raw state-dict and wrapped checkpoints
if isinstance(ckpt, dict):
    state_dict = ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt))
else:
    state_dict = ckpt

# Strip 'model.' prefix if it was saved from a Lightning wrapper
if any(k.startswith('model.') for k in state_dict):
    state_dict = {k[len('model.'):]: v for k, v in state_dict.items()}

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"Checkpoint loaded. Missing keys: {len(missing)}, Unexpected: {len(unexpected)}")
if missing:
    print("  Missing (first 5):", missing[:5])

model.eval().to(DEVICE)

# Quick sanity check — dummy forward pass
with torch.no_grad():
    dummy = torch.zeros(1, 1, N_MELS, 313).to(DEVICE)
    out   = model(dummy)
print(f"Forward pass OK — output shape: {out.shape}")  # should be (1, 234)

## 5 — Run inference on all unlabeled files

In [ ]:
from tqdm.notebook import tqdm

BATCH_SIZE = 32   # chunks per forward pass; reduce to 16 if OOM

def get_chunk_starts(filepath):
    """Return list of 5-second window start times for a soundscape file."""
    try:
        duration = librosa.get_duration(path=filepath)
    except Exception:
        return []
    return list(range(0, int(duration), DURATION))


@torch.no_grad()
def predict_file(filepath):
    """Return a list of dicts — one per 5-second chunk.
    Each dict has: filename, start_sec, end_sec, probs (np.ndarray, shape NUM_CLASSES)
    """
    fname   = os.path.basename(filepath)
    starts  = get_chunk_starts(filepath)
    results = []

    # Process in mini-batches for speed
    for i in range(0, len(starts), BATCH_SIZE):
        batch_starts = starts[i : i + BATCH_SIZE]
        specs = []
        valid_starts = []

        for s in batch_starts:
            spec = load_chunk(filepath, s)
            if spec is not None:
                specs.append(spec)
                valid_starts.append(s)

        if not specs:
            continue

        batch = torch.stack(specs).to(DEVICE)   # (B, 1, N_MELS, time)
        probs = torch.sigmoid(model(batch)).cpu().numpy()  # (B, NUM_CLASSES)

        for s, p in zip(valid_starts, probs):
            results.append({
                'filename':  fname,
                'start_sec': s,
                'end_sec':   s + DURATION,
                'probs':     p,        # raw probabilities — kept for soft-label training
            })

    return results


# ── Main inference loop ──────────────────────────────────────────────────────
all_results = []

for fname in tqdm(unlabeled_files, desc='Pseudo-labeling files'):
    fpath = os.path.join(SS_DIR, fname)
    try:
        rows = predict_file(fpath)
        all_results.extend(rows)
    except Exception as e:
        print(f"ERROR on {fname}: {e}")

print(f"\nTotal 5-second chunks processed: {len(all_results)}")

## 6 — Save raw probabilities

Saved as a `.npz` (compact) + a CSV with `max_prob` and top-3 labels for quick inspection.

In [ ]:
# Build a metadata DataFrame (no probs yet)
meta_rows = [{'filename': r['filename'],
              'start_sec': r['start_sec'],
              'end_sec':   r['end_sec']} for r in all_results]
meta_df = pd.DataFrame(meta_rows)

# Stack probabilities into a 2-D array (N_chunks, NUM_CLASSES)
probs_array = np.stack([r['probs'] for r in all_results])   # (N, 234)

# Save raw probs as npz — fast to reload, compact
npz_path = os.path.join(OUT_DIR, 'pseudo_probs.npz')
np.savez_compressed(npz_path,
                    probs=probs_array,
                    filenames=meta_df['filename'].values,
                    start_secs=meta_df['start_sec'].values)
print(f"Raw probs saved → {npz_path}  shape: {probs_array.shape}")

# Also save a human-readable CSV with max_prob and top-3 species
top3_indices = np.argsort(probs_array, axis=1)[:, -3:][:, ::-1]
meta_df['max_prob']  = probs_array.max(axis=1)
meta_df['top1_label'] = [idx2label[i[0]] for i in top3_indices]
meta_df['top2_label'] = [idx2label[i[1]] for i in top3_indices]
meta_df['top3_label'] = [idx2label[i[2]] for i in top3_indices]
meta_df['top1_prob']  = probs_array[np.arange(len(probs_array)), top3_indices[:, 0]]

raw_csv = os.path.join(OUT_DIR, 'pseudo_labels_raw.csv')
meta_df.to_csv(raw_csv, index=False)
print(f"Raw CSV saved  → {raw_csv}")

meta_df.head(10)

## 7 — Inspect confidence distribution

Look at this plot **before** choosing your threshold.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution of max-prob per chunk
axes[0].hist(meta_df['max_prob'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(0.3, color='red',    linestyle='--', label='threshold=0.30')
axes[0].axvline(0.5, color='orange', linestyle='--', label='threshold=0.50')
axes[0].set_title('Max-prob distribution (per 5-second chunk)')
axes[0].set_xlabel('Max predicted probability')
axes[0].set_ylabel('Count')
axes[0].legend()

# How many chunks survive each threshold
thresholds = np.linspace(0.1, 0.9, 50)
survival   = [(meta_df['max_prob'] >= t).sum() for t in thresholds]
axes[1].plot(thresholds, survival, color='steelblue')
axes[1].axvline(0.3, color='red',    linestyle='--')
axes[1].axvline(0.5, color='orange', linestyle='--')
axes[1].set_title('Chunks surviving threshold')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('N chunks')

plt.tight_layout()
plt.show()

for t in [0.2, 0.3, 0.4, 0.5, 0.6]:
    n = (meta_df['max_prob'] >= t).sum()
    print(f"  threshold={t:.1f} → {n:5d} chunks  ({100*n/len(meta_df):.1f}%)")

## 8 — Apply threshold → hard labels

In [ ]:
# ── Tune this after looking at the plot above ────────────────────────────────
THRESHOLD = 0.35   # per-class probability cutoff
MIN_MAX_PROB = 0.35  # drop chunks where even the top class is below this
# ─────────────────────────────────────────────────────────────────────────────

hard_labels = []
for i, row in enumerate(all_results):
    p = row['probs']  # (NUM_CLASSES,)

    # Skip low-confidence chunks entirely
    if p.max() < MIN_MAX_PROB:
        continue

    # All classes above threshold
    detected = [idx2label[k] for k, prob in enumerate(p) if prob >= THRESHOLD]

    if not detected:
        continue

    hard_labels.append({
        'filename':     row['filename'],
        'start_sec':    row['start_sec'],
        'end_sec':      row['end_sec'],
        'primary_label': ';'.join(detected),
        'max_prob':     float(p.max()),
        'n_labels':     len(detected),
        'is_pseudo':    True,
    })

pseudo_df = pd.DataFrame(hard_labels)
print(f"Chunks kept after threshold={THRESHOLD}: {len(pseudo_df)} / {len(all_results)}")
print(f"Average labels per chunk: {pseudo_df['n_labels'].mean():.2f}")
print()
pseudo_df.head()

In [ ]:
# Label frequency check — make sure one class isn't dominating
from collections import Counter

counter = Counter()
for row in pseudo_df['primary_label']:
    counter.update(row.split(';'))

top20 = counter.most_common(20)
labels_top, counts_top = zip(*top20)

plt.figure(figsize=(14, 4))
plt.bar(labels_top, counts_top, color='steelblue')
plt.title('Top-20 species in pseudo-labels')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"Total unique pseudo-labeled species: {len(counter)}")

## 9 — Save filtered pseudo labels

In [ ]:
filtered_path = os.path.join(OUT_DIR, 'pseudo_labels_filtered.csv')
pseudo_df.to_csv(filtered_path, index=False)
print(f"Filtered pseudo labels saved → {filtered_path}")

## 10 — Merge with ground-truth labeled soundscapes

Produces `pseudo_labels_for_training.csv` — a drop-in replacement for
`labeled_soundscapes_split.csv`. Pass it to `SoundscapeChunkDataset` as-is.

> **Fold assignment:** the pseudo-labeled rows get `fold=-1` so they are
> always included in training but never leak into validation.
> Swap to a real fold value if you want to validate on pseudo data.

In [ ]:
# Ground-truth labeled soundscapes (from your labeled_soundscapes_split.csv)
gt_df = pd.read_csv(LABELS_CSV).copy()
gt_df['is_pseudo'] = False

# Align columns: pseudo_df must have same key columns as gt_df
# gt_df columns typically: filename, start_sec, end_sec, primary_label, fold
# Check:
print("GT columns  :", gt_df.columns.tolist())
print("Pseudo cols :", pseudo_df.columns.tolist())

In [ ]:
# Assign pseudo rows to fold=-1 so they go to train-only, never val
pseudo_for_merge = pseudo_df[['filename', 'start_sec', 'end_sec',
                              'primary_label', 'is_pseudo']].copy()
pseudo_for_merge['fold'] = -1   # always train; excluded from val splits

# Combine
combined_df = pd.concat([gt_df, pseudo_for_merge], ignore_index=True)

train_path = os.path.join(OUT_DIR, 'pseudo_labels_for_training.csv')
combined_df.to_csv(train_path, index=False)

print(f"Ground-truth rows : {len(gt_df)}")
print(f"Pseudo-label rows : {len(pseudo_for_merge)}")
print(f"Combined total    : {len(combined_df)}")
print(f"Saved → {train_path}")
combined_df.head()

## 11 — How to use in training

In your training notebook, swap in the combined CSV:

```python
# OLD
ss_df = pd.read_csv(f'{UTILS}/labeled_soundscapes_split.csv')

# NEW — includes pseudo-labeled rows (fold=-1 rows always land in train)
ss_df = pd.read_csv(f'{OUT_DIR}/pseudo_labels_for_training.csv')

# In build_loaders, the fold filter already handles this correctly:
#   fold != 0  includes fold=-1 rows  → they go into train ✓
#   fold == 0  excludes fold=-1 rows  → they stay out of val ✓
def build_loaders(fold, ...):
    train_ss = SoundscapeChunkDataset(
        ss_df[ss_df['fold'] != fold],   # -1 != 0 → included in train ✓
        audio_dir=SS_DIR, label2idx=label2idx, mode='train',
    )
    val_ss = SoundscapeChunkDataset(
        ss_df[ss_df['fold'] == fold],   # -1 != 0 → excluded from val ✓
        audio_dir=SS_DIR, label2idx=label2idx, mode='val',
    )
    ...
```

### Optional — soft-label training

Instead of hard binary targets, you can pass the raw probabilities as soft
labels. This avoids the hard threshold step entirely and tends to work
better when the model is not very confident:

```python
data = np.load('pseudo_probs.npz')
probs      = data['probs']       # (N, 234)
filenames  = data['filenames']
start_secs = data['start_secs']

# In __getitem__, use the stored prob vector directly as the label tensor
# instead of building a one-hot from the species code string.
label = torch.from_numpy(probs[idx]).float()
```